#Avocado Price - Model Building

Train and compare 7 regression models. ~ 18k rows x ~ 70 columns (after region one-hot encoding) - moderate scale, comfortable for full grid-search

#1. Imports & Load

In [ ]:
%%writefile utils.py
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score # Import cross_val_score
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd # Import pandas for DataFrame

def evaluate_model(model_name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)

    print(f"--- {model_name} Evaluation ---")
    print(f"MAE: {mae:.4f}")
    print(f"MSE: {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R-squared: {r2:.4f}")
    print("-" * (len(model_name) + 16))

    # Return a dictionary of metrics for potential comparison later
    return {"model": model_name, "MAE": mae, "MSE": mse, "RMSE": rmse, "R2": r2}

def plot_actual_vs_predicted(y_true, y_pred, title, ax):
    ax.scatter(y_true, y_pred, alpha=0.6)
    ax.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'k--', lw=2)
    ax.set_xlabel("Actual Values")
    ax.set_ylabel("Predicted Values")
    ax.set_title(f"{title}: Actual vs. Predicted")
    ax.grid(True)

def plot_residuals(y_true, y_pred, title, ax):
    residuals = y_true - y_pred
    ax.scatter(y_pred, residuals, alpha=0.6)
    ax.hlines(0, y_pred.min(), y_pred.max(), colors='r', linestyles='--')
    ax.set_xlabel("Predicted Values")
    ax.set_ylabel("Residuals")
    ax.set_title(f"{title}: Residuals Plot")
    ax.grid(True)

def cross_validate_model(model, X, y, cv=5, scoring='r2'): # Added scoring parameter
    print(f"Performing {cv}-fold cross-validation for {model.__class__.__name__} with scoring='{scoring}'...")
    scores = cross_val_score(model, X, y, cv=cv, scoring=scoring, n_jobs=-1) # Use cross_val_score
    print(f"  Mean {scoring.upper()} score: {np.mean(scores):.4f}")
    print(f"  Std of {scoring.upper()} scores: {np.std(scores):.4f}")
    return {'mean_score': np.mean(scores), 'std_score': np.std(scores), 'scores': scores.tolist()}

def compare_models(models_metrics):
    # Create a DataFrame from the list of model metrics
    df_metrics = pd.DataFrame(models_metrics)
    # Set 'model' as index and sort by R2 score (or any other preferred metric)
    df_metrics = df_metrics.set_index('model').sort_values(by='R2', ascending=False)
    print("Model Comparison (Sorted by R2 Score):")
    print(df_metrics.round(4))
    return df_metrics # Return the DataFrame

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, AdaBoostClassifier, GradientBoostingClassifier, GradientBoostingRegressor

import sys
sys.path.append(".")
import importlib
import utils
importlib.reload(utils)
from utils import (evaluate_model, plot_actual_vs_predicted, plot_residuals,
                   cross_validate_model, compare_models)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

%matplotlib inline

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
df = pd.read_csv("avocado_cleaned.csv")
df = df.drop(columns=['Date'])
print(f"Shape: {df.shape}")
df.head()

#2. Train / Test Split + Scaling

In [ ]:
y = df['AveragePrice']
X = df.drop(columns=['AveragePrice'])

# Identify categorical columns
categorical_cols = X.select_dtypes(include=['object']).columns

# Apply one-hot encoding to categorical features
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

print("Shape after one-hot encoding:", X.shape)
display(X.head())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f"Train:  {X_train.shape}, Test:  {X_test.shape}")
print(f"Train mean price:  ${y_train.mean():.2f}, Test mean:  ${y_test.mean():.2f}")

#3. Model 1 - Linear Regression

In [ ]:
lr = LinearRegression()
lr.fit(X_train_s, y_train)
pred_lr = lr.predict(X_test_s)
m_lr = evaluate_model("Linear Regression", y_test, pred_lr)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
plot_actual_vs_predicted(y_test, pred_lr, "Linear Regression", ax=axes[0])
plot_residuals(y_test, pred_lr, "Linear Regression", ax=axes[1])
plt.tight_layout(); plt.show()

#4. Model 2 - Ridge

In [ ]:
ridge  = Ridge(alpha=1.0, random_state=42)
ridge.fit(X_train_s, y_train)
pred_ridge = ridge.predict(X_test_s)
m_ridge = evaluate_model("Ridge", y_test, pred_ridge)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
plot_actual_vs_predicted(y_test, pred_ridge, "Ridge", ax=axes[0])
plot_residuals(y_test, pred_ridge, "Ridge", ax=axes[1])
plt.tight_layout(); plt.show()


#5. Model 3 - Lasso

In [ ]:
lasso = Lasso(alpha=0.001, random_state=42, max_iter=20000)
lasso.fit(X_train_s, y_train)
pred_lasso = lasso.predict(X_test_s)
m_lasso = evaluate_model("Lasso", y_test, pred_lasso)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
plot_actual_vs_predicted(y_test, pred_lasso, "Lasso", ax=axes[0])
plot_residuals(y_test, pred_lasso, "Lasso", ax=axes[1])
plt.tight_layout(); plt.show()

#6. Model 4 - Decision Tree

In [ ]:
dt = DecisionTreeRegressor(random_state=42, max_depth=10)
dt.fit(X_train, y_train)
pred_dt = dt.predict(X_test)
m_dt = evaluate_model("Decision Tree", y_test, pred_dt)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
plot_actual_vs_predicted(y_test, pred_dt, "Decision Tree", ax=axes[0])
plot_residuals(y_test, pred_dt, "Decision Tree", ax=axes[1])
plt.tight_layout(); plt.show()

#7. Model 5 - Random Forest

In [ ]:
rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)
m_rf = evaluate_model("Random Forest", y_test, pred_rf)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
plot_actual_vs_predicted(y_test, pred_rf, "Random Forest", ax=axes[0])
plot_residuals(y_test, pred_rf, "Random Forest", ax=axes[1])
plt.tight_layout(); plt.show()

#8. Model 6 - Gradient Boosting

In [ ]:
gb = GradientBoostingRegressor(n_estimators=200, random_state=42)
gb.fit(X_train, y_train)
pred_gb = gb.predict(X_test)
m_gb = evaluate_model("Gradient Boosting", y_test, pred_gb)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
plot_actual_vs_predicted(y_test, pred_gb, "Gradient Boosting", ax=axes[0])
plot_residuals(y_test, pred_gb, "Gradient Boosting", ax=axes[1])
plt.tight_layout(); plt.show()

#9. Model 7 - KNN (with K optimization)

In [ ]:
ks = [3, 5, 7, 11, 15, 21]
r2s = []
for k in ks:
    kk = KNeighborsRegressor(n_neighbors=k, n_jobs=-1) # Changed to KNeighborsRegressor
    kk.fit(X_train_s, y_train)
    r2s.append(kk.score(X_test_s, y_test))

best_k = ks[int(np.argmax(r2s))]
plt.figure(figsize=(8, 4))
plt.plot(ks, r2s, marker="o")
plt.axvline(best_k, color="red", linestyle="--", label=f"Best k={best_k}")
plt.xlabel("k"); plt.ylabel("Test r2"); plt.title("KNN - R2 vs K") # Fixed typo plt.xlabe to plt.xlabel
plt.legend(); plt.tight_layout(); plt.show()

knn = KNeighborsRegressor(n_neighbors=best_k, n_jobs=-1)
knn.fit(X_train_s, y_train)
pred_knn = knn.predict(X_test_s)
m_knn = evaluate_model("KNN", y_test, pred_knn)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
plot_actual_vs_predicted(y_test, pred_knn, f"KNN", ax=axes[0])
plot_residuals(y_test, pred_knn, f"KNN (k={best_k})", ax=axes[1])
plt.tight_layout(); plt.show()

#10. Feature Importance (tree-based)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, model, name in [(axes[0], rf, "Random Forest"), (axes[1], gb, "Gradient Boosting")]:
    imp = pd.Series(model.feature_importances_, index=X.columns).sort_values()
    imp.tail(15).plot(kind='barh', ax=ax, color='teal')
    ax.set_title(f"Top 15 features - {name}")
plt.tight_layout(); plt.show()

#11. Model Comparison

In [ ]:
results = [m_lr, m_ridge, m_lasso, m_dt, m_rf, m_gb, m_knn]
df_results = compare_models(results)
df_results.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
df_results[["R2"]].plot(kind='bar', ax=ax, color="teal", legend=False)
ax.set_title("Model comparison (R2)")
plt.xticks(rotation=30, ha="right"); plt.tight_layout(); plt.show()

#12. 5-Fold Cross-Validation

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

import sys
sys.path.append(".")
import importlib
import utils
importlib.reload(utils)
from utils import (evaluate_model, plot_actual_vs_predicted, plot_residuals,
                   cross_validate_model, compare_models)

# Data Loading and Preprocessing (to ensure X_train, y_train, X_train_s are defined)
df = pd.read_csv("avocado_cleaned.csv")
df = df.drop(columns=['Date'])
y = df['AveragePrice']
X = df.drop(columns=['AveragePrice'])
categorical_cols = X.select_dtypes(include=['object']).columns
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test) # Needed for best_k calculation

# Model Instantiations
lr = LinearRegression()
ridge = Ridge(alpha=1.0, random_state=42)
lasso = Lasso(alpha=0.001, random_state=42, max_iter=20000)
dt = DecisionTreeRegressor(random_state=42, max_depth=10)
rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
gb = GradientBoostingRegressor(n_estimators=200, random_state=42)

# KNN and best_k calculation (copied from earlier cell)
ks = [3, 5, 7, 11, 15, 21]
r2s = []
for k in ks:
    kk = KNeighborsRegressor(n_neighbors=k, n_jobs=-1)
    kk.fit(X_train_s, y_train)
    r2s.append(kk.score(X_test_s, y_test))
best_k = ks[int(np.argmax(r2s))]
knn = KNeighborsRegressor(n_neighbors=best_k, n_jobs=-1)

cv_models = {
    "Linear Regression": (lr, X_train_s),
    "Ridge": (ridge, X_train_s),
    "Lasso": (lasso, X_train_s),
    "Decision Tree": (dt, X_train),
    "Random Forest": (rf, X_train),
    "Gradient Boosting": (gb, X_train),
    f"KNN (K={best_k})": (knn, X_train_s),
}
cv_scores = {}
for name, (m, X_in) in cv_models.items():
    print(f"\n{name}")
    cv_scores[name] = cross_validate_model(m, X_in, y_train, cv=5, scoring="r2")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

cv_df_boxplot = pd.DataFrame({name: metrics['scores'] for name, metrics in cv_scores.items()})
plt.figure(figsize=(12, 5))
sns.boxplot(data=cv_df_boxplot, palette="Set2")
plt.title("5-fold CV R2 distribution")
plt.xticks(rotation=30, ha='right'); plt.ylabel("R2")
plt.tight_layout(); plt.show()

#13. Hyperparameter Tuning - Random Forest (small grid)

In [ ]:
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.ensemble import RandomForestRegressor
import pandas as pd
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("avocado_cleaned.csv")
df = df.drop(columns=['Date'])
y = df['AveragePrice']
X = df.drop(columns=['AveragePrice'])

categorical_cols = X.select_dtypes(include=['object']).columns
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Reduced search space for faster execution
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [10, 20],
    'min_samples_leaf': [1, 2],
}

# Consider reducing cv to 2 if it's still too slow, e.g., cv=2
grid = GridSearchCV(RandomForestRegressor(random_state=42, n_jobs=-1),
                    param_grid, cv=3, scoring='r2', n_jobs=-1)
grid.fit(X_train, y_train)
print(f"Best parameters: {grid.best_params_}")
print(f"Best CV R2: {grid.best_score_:.4f}")

In [ ]:
import importlib
import utils
importlib.reload(utils)
from utils import evaluate_model, plot_actual_vs_predicted, plot_residuals # Re-import functions

if hasattr(grid, 'best_estimator_'):
    rf_tuned = grid.best_estimator_
else:
    print("Warning: GridSearchCV was not completed successfully. Using the untuned Random Forest model 'rf' as a fallback.")
    # Fit the untuned Random Forest model if it's not already fitted
    if not hasattr(rf, 'n_features_in_') or rf.n_features_in_ is None:
        print("Fitting the untuned Random Forest model 'rf'.")
        rf.fit(X_train, y_train)
    rf_tuned = rf # Fallback to the untuned Random Forest model

pred_rf_tuned = rf_tuned.predict(X_test)
m_rf_tuned = evaluate_model("Random Forest (tuned)", y_test, pred_rf_tuned)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
plot_actual_vs_predicted(y_test, pred_rf_tuned, "Random Forest (tuned)", ax=axes[0])
plot_residuals(y_test, pred_rf_tuned, "Random Forest (tuned)", ax=axes[1])
plt.tight_layout(); plt.show()

#14. Tuned Feature Importance

In [ ]:
imp = pd.Series(rf_tuned.feature_importances_, index=X.columns).sort_values()
plt.figure(figsize=(8, 7))
imp.tail(15).plot(kind='barh', color='teal')
plt.title("Top 15 features - Random Forest (tuned)")
plt.tight_layout(); plt.show()

#15. Prediction Example

In [ ]:
sample = X_test.iloc[[0, 1, 2]]
preds = rf_tuned.predict(sample)
for i, (idx, row) in enumerate(sample.iterrows()):
  actual = y_test.loc[idx]
  print(f"Sample {i} actual=${actual:>5.2f}  Predicted: ${preds[i]:>5.2f}   error=${preds[i]-actual:+.2f}")


#16. Final Summary

| Aspect | Result
| :---   | :----   |
| **Best baseline**  | Random Forest / Gradient Bosting ($R^2 \approx$ 0.88-0.91)  |
| **Tuned Model**  | Random Forest tuned via `GridSearchCV`    |
| **Top Features** | `type_organic`, `year`, `region_*`, `log_total_volume`, `month`  |
| **Caveats**  | Random splits igore the time-series structure of weekly data for production-use, prefer time-series aware CV.

#Next Steps

**.** Use **TimeSeriesSplit** to honor the weekly observation order.

**.** Try **per-region models** (one per market) - region effects are large.

**.** Add **rolling averages** of price and volume from the previous N weeks as features.

**.** Investigate **2017 price spike** mechanics (industry-level supply shock).